# Conformal TB Triage: Lung Segmentation Sensitivity (§5.4.17)

**Runtime: GPU T4 (Colab free tier)**

Re-run of the lung segmentation sensitivity analysis from Notebook 03.
The original run silently fell back to an intensity-based mask that was
effectively a no-op, producing identical whole-image and lung-only metrics.
(`lungmask` R231 is a CT model that returns empty masks on 2D CXR images.)

This notebook uses **classical CV** (CLAHE + Otsu + connected components +
convex hull) for CXR lung field segmentation:
1. Validates segmentation visually on 4 images before proceeding
2. Segments n=1000 test images (same subset/seed as Notebook 03)
3. Compares whole-image vs lung-only vs mediastinal-only embeddings
4. Overwrites `lung_segmentation.csv` on Drive

In [ ]:
# ── Cell 1: Install & verify dependencies ────────────────────────────
import subprocess, sys

packages = [
    'torch', 'torchvision',
    'transformers',
    'pandas', 'pyarrow',
    'Pillow',
    'tqdm',
    'scikit-learn',
    'opencv-python-headless',
]
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('pip stdout:', result.stdout[-2000:])
    print('pip stderr:', result.stderr[-2000:])
    raise RuntimeError('pip install failed — check output above')

import cv2
print(f'OpenCV {cv2.__version__}')
print('Packages installed.')

In [ ]:
# ── Cell 2: Setup ────────────────────────────────────────────────────
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import normalize
from sklearn.isotonic import IsotonicRegression
from transformers import AutoModel, AutoImageProcessor
import gc

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'GPU required — change runtime to GPU'
print(f'GPU: {torch.cuda.get_device_name(0)} '
      f'({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)', flush=True)

DRIVE_ROOT = Path('/content/drive/MyDrive/conformal-tb-triage')
EMB_DIR    = DRIVE_ROOT / 'data' / 'interim' / 'embeddings'
RAW        = DRIVE_ROOT / 'data' / 'raw'
RESULTS_DIR = DRIVE_ROOT / 'results' / 'tables'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

# Load splits and baseline embeddings
splits = pd.read_parquet(DRIVE_ROOT / 'data' / 'processed' / 'splits.parquet')
baseline = pd.read_parquet(EMB_DIR / 'rad_dino.parquet')
emb_cols = [c for c in baseline.columns if c.startswith('emb_')]
print(f'Splits: {len(splits)}, Baseline embeddings: {len(baseline)}', flush=True)

In [ ]:
# ── Cell 3: Load RAD-DINO + build baseline pipeline ─────────────────
print('Loading RAD-DINO...', flush=True)
processor = AutoImageProcessor.from_pretrained('microsoft/rad-dino', use_fast=False)
model = AutoModel.from_pretrained('microsoft/rad-dino').to(device)
model.eval()
print('RAD-DINO loaded.', flush=True)


def extract_batch(images, batch_size=64):
    """Extract RAD-DINO CLS embeddings from a list of PIL images."""
    all_emb = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i + batch_size]
        inputs = processor(images=batch, return_tensors='pt')['pixel_values'].to(device)
        with torch.no_grad():
            emb = model(pixel_values=inputs).last_hidden_state[:, 0, :]
        all_emb.append(emb.cpu().float().numpy())
    return np.vstack(all_emb)


def get_baseline_pipeline():
    """Train probe + isotonic on calibration/dev splits; return components."""
    df = baseline[baseline['tb_binary'].isin(['tb_positive', 'tb_negative'])].copy()
    X = normalize(df[emb_cols].values.astype(np.float32), norm='l2')
    y = (df['tb_binary'] == 'tb_positive').astype(int).values
    sp = df['split'].values

    cal_m = sp == 'calibration'
    dev_m = sp == 'dev'
    test_m = sp == 'test'

    probe = LogisticRegression(C=10, max_iter=2000, solver='lbfgs', random_state=SEED)
    probe.fit(X[cal_m], y[cal_m])

    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(probe.predict_proba(X[dev_m])[:, 1], y[dev_m])

    cal_prob = iso.predict(probe.predict_proba(X[cal_m])[:, 1])
    cal_y = y[cal_m]
    baseline_auroc = roc_auc_score(y[test_m], probe.predict_proba(X[test_m])[:, 1])
    return probe, iso, cal_prob, cal_y, baseline_auroc


def evaluate_new_embeddings(new_X, y, alpha=0.10):
    """Run probe + isotonic + Mondrian conformal on new embeddings."""
    X_norm = normalize(new_X.astype(np.float32), norm='l2')
    raw_prob = probe.predict_proba(X_norm)[:, 1]
    prob = iso.predict(raw_prob)
    auroc = roc_auc_score(y, raw_prob) if len(np.unique(y)) > 1 else float('nan')

    # Mondrian conformal thresholds
    cal_sc = np.where(cal_y == 1, 1 - cal_prob, cal_prob)
    th = {}
    for c in [0, 1]:
        cs = cal_sc[cal_y == c]
        n = len(cs)
        th[c] = np.quantile(cs, min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0))
    sets = []
    for p in prob:
        s = set()
        if (1 - p) <= th.get(1, 1.0): s.add(1)
        if p <= th.get(0, 1.0): s.add(0)
        sets.append(s)

    cov = [int(yi in s) for yi, s in zip(y, sets)]
    tb_m = y == 1
    tb_cov = np.mean([cov[i] for i in range(len(cov)) if tb_m[i]]) if tb_m.any() else float('nan')
    singleton = np.mean([len(s) == 1 for s in sets])
    return {'auroc': round(auroc, 4), 'tb_cov': round(tb_cov, 4), 'singleton': round(singleton, 4)}


probe, iso, cal_prob, cal_y, baseline_auroc = get_baseline_pipeline()
print(f'Baseline AUROC: {baseline_auroc:.4f}')

In [ ]:
# ── Cell 4: Copy test images to local disk ───────────────────────────
import os, shutil
LOCAL_IMG = Path('/content/local_images')

existing = (
    sum(1 for _ in LOCAL_IMG.rglob('*.png')) + sum(1 for _ in LOCAL_IMG.rglob('*.jpg'))
    if LOCAL_IMG.exists() else 0
)
if existing >= len(splits) * 0.9:
    print(f'Already cached ({existing} files). Skipping copy.', flush=True)
else:
    for name, src_dir in [
        ('shenzhen_montgomery', RAW / 'shenzhen_montgomery'),
        ('tbx11k', RAW / 'tbx11k'),
        ('chexpert_subset', RAW / 'chexpert_subset'),
        ('mendeley_pakistan', RAW / 'mendeley_pakistan'),
    ]:
        dst = LOCAL_IMG / name
        if dst.exists() and (any(dst.rglob('*.png')) or any(dst.rglob('*.jpg'))):
            print(f'{name}: cached', flush=True)
            continue
        if not src_dir.exists():
            print(f'{name}: source missing', flush=True)
            continue
        print(f'{name}: copying...', flush=True)
        os.system(f'cp -r "{src_dir}" "{dst}"')
        print(f'  done', flush=True)


def remap(row):
    p = Path(row['file_path'])
    for ds_key, local_name in [
        ('shenzhen', 'shenzhen_montgomery'),
        ('montgomery', 'shenzhen_montgomery'),
        ('tbx11k', 'tbx11k'),
        ('chexpert', 'chexpert_subset'),
        ('pakistan', 'mendeley_pakistan'),
    ]:
        if row.get('dataset', '') == ds_key:
            local = LOCAL_IMG / local_name
            if local.exists():
                matches = list(local.rglob(p.name))
                if matches:
                    return str(matches[0])
    return row['file_path']


splits['file_path'] = splits.apply(remap, axis=1)
n_local = splits['file_path'].str.startswith('/content/local').sum()
print(f'Remapped: {n_local} local, {len(splits) - n_local} Drive', flush=True)

In [ ]:
# ── Cell 5: Load test images (same subset as Notebook 03) ────────────
test_df = splits[
    (splits['split'] == 'test') &
    (splits['tb_binary'].isin(['tb_positive', 'tb_negative']))
].copy()

test_images, test_y, test_pids = [], [], []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Loading images'):
    try:
        img = Image.open(row['file_path']).convert('RGB')
        test_images.append(img)
        test_y.append(1 if row['tb_binary'] == 'tb_positive' else 0)
        test_pids.append(row['patient_id'])
    except Exception:
        pass
test_y = np.array(test_y)
print(f'Loaded {len(test_images)} test images '
      f'({test_y.sum()} TB+, {(test_y == 0).sum()} TB-)', flush=True)

# Same n=1000 subset and seed as Notebook 03
n_seg = min(1000, len(test_images))
rng = np.random.RandomState(SEED)
seg_idx = rng.choice(len(test_images), n_seg, replace=False)
seg_images = [test_images[i] for i in seg_idx]
seg_y = test_y[seg_idx]
print(f'Subset: {n_seg} images ({seg_y.sum()} TB+, {(seg_y == 0).sum()} TB-)')

In [ ]:
# ── Cell 6: CXR lung segmentation & sanity check ────────────────────
# lungmask R231 is a CT model — it returns empty masks on 2D CXR images.
# Instead, use classical CV: Otsu thresholding + morphology + connected
# components. This is the standard approach for CXR lung field extraction
# and doesn't require a model trained on a different imaging modality.

import cv2
import matplotlib.pyplot as plt


def segment_lung(pil_img):
    """Segment lung fields from a PA/AP chest X-ray using classical CV.

    Pipeline:
      1. Grayscale → Gaussian blur (reduce noise)
      2. CLAHE (normalise contrast across heterogeneous exposures)
      3. Otsu threshold (lungs are the largest dark region)
      4. Morphological close → open (fill gaps, remove small noise)
      5. Select the two largest connected components (L + R lung)
      6. Convex-hull fill per component (recover rib-occluded regions)
    """
    gray = np.array(pil_img.convert('L'))
    h, w = gray.shape

    # 1. Blur
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)

    # 2. CLAHE to handle varying exposure / contrast
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(blurred)

    # 3. Otsu threshold — lungs appear dark (low intensity) in CXR
    _, binary = cv2.threshold(enhanced, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # 4. Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

    # 5. Connected components — keep two largest (left + right lung)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )
    if num_labels <= 1:
        return np.zeros((h, w), dtype=np.uint8)

    # Sort by area descending, skip background (label 0)
    areas = stats[1:, cv2.CC_STAT_AREA]
    sorted_idx = np.argsort(areas)[::-1] + 1

    mask = np.zeros((h, w), dtype=np.uint8)
    min_area = 0.03 * h * w  # component must be ≥3% of image
    for i in range(min(2, len(sorted_idx))):
        idx = sorted_idx[i]
        if stats[idx, cv2.CC_STAT_AREA] >= min_area:
            mask[labels == idx] = 1

    # 6. Convex hull per component to fill rib-shadowed gaps
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    hull_mask = np.zeros_like(mask)
    for cnt in contours:
        hull = cv2.convexHull(cnt)
        cv2.drawContours(hull_mask, [hull], -1, 1, thickness=cv2.FILLED)

    return hull_mask


# ── Sanity check on 4 diverse images ────────────────────────────────
check_idx = [0, n_seg // 4, n_seg // 2, 3 * n_seg // 4]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
mask_fracs = []

for col, ci in enumerate(check_idx):
    img = seg_images[ci]
    mask = segment_lung(img)
    frac = mask.mean()
    mask_fracs.append(frac)

    rgb = np.array(img)
    overlay = rgb.copy()
    overlay[mask == 0] = (overlay[mask == 0] * 0.3).astype(np.uint8)

    axes[0, col].imshow(img)
    axes[0, col].set_title(f'Original (idx={ci})')
    axes[0, col].axis('off')

    axes[1, col].imshow(mask, cmap='gray')
    axes[1, col].set_title(f'Lung mask ({frac:.1%})')
    axes[1, col].axis('off')

    axes[2, col].imshow(overlay)
    axes[2, col].set_title('Overlay')
    axes[2, col].axis('off')

plt.suptitle('Lung Segmentation Sanity Check (Classical CV)', fontsize=14)
plt.tight_layout()
plt.show()

for ci, frac in zip(check_idx, mask_fracs):
    print(f'  Image {ci}: lung fraction = {frac:.3f}')

min_frac, max_frac = min(mask_fracs), max(mask_fracs)
assert min_frac > 0.05, (
    f'Masks are too small (min {min_frac:.1%}). '
    f'Segmentation is not finding lung regions.'
)
assert max_frac < 0.85, (
    f'Masks are too large (max {max_frac:.1%}). '
    f'Segmentation is returning near-trivial masks.'
)
print(f'\nSanity check PASSED (lung fraction range: {min_frac:.1%} – {max_frac:.1%})')

In [ ]:
# ── Cell 7: Segment all 1000 images ──────────────────────────────────
# Build three image variants:
#   (a) whole_image   — original, unmodified
#   (b) lung_only     — non-lung pixels zeroed
#   (c) mediastinal   — lung pixels zeroed, mediastinum/periphery kept

lung_only_images = []
mediastinal_images = []
lung_fractions = []
seg_failures = 0

for i, img in enumerate(tqdm(seg_images, desc='Segmenting')):
    try:
        mask = segment_lung(img)
        frac = mask.mean()
        lung_fractions.append(frac)

        # Degenerate mask check (per-image)
        if frac < 0.02 or frac > 0.98:
            # Mask is degenerate for this image — keep original for both
            lung_only_images.append(img)
            mediastinal_images.append(img)
            seg_failures += 1
            continue

        rgb = np.array(img)
        mask_3ch = mask[:, :, np.newaxis]

        # Lung-only: zero non-lung
        lung_rgb = rgb * mask_3ch
        lung_only_images.append(Image.fromarray(lung_rgb))

        # Mediastinal: zero lung, keep rest
        med_rgb = rgb * (1 - mask_3ch)
        mediastinal_images.append(Image.fromarray(med_rgb))

    except Exception as e:
        # Segmentation failed — use original as fallback
        lung_only_images.append(img)
        mediastinal_images.append(img)
        lung_fractions.append(float('nan'))
        seg_failures += 1

lung_fractions = np.array(lung_fractions)
print(f'\nSegmentation complete:')
print(f'  Successful: {n_seg - seg_failures}/{n_seg}')
print(f'  Failures/degenerate: {seg_failures}')
print(f'  Mean lung fraction: {np.nanmean(lung_fractions):.3f}')
print(f'  Median: {np.nanmedian(lung_fractions):.3f}')
print(f'  Range: [{np.nanmin(lung_fractions):.3f}, {np.nanmax(lung_fractions):.3f}]')

assert seg_failures < n_seg * 0.1, (
    f'Too many segmentation failures ({seg_failures}/{n_seg}). '
    f'Results would not be meaningful.'
)

In [ ]:
# ── Cell 8: Extract embeddings & evaluate ────────────────────────────

print('Extracting whole-image embeddings...', flush=True)
whole_emb = extract_batch(seg_images)

print('Extracting lung-only embeddings...', flush=True)
lung_emb = extract_batch(lung_only_images)

print('Extracting mediastinal-only embeddings...', flush=True)
med_emb = extract_batch(mediastinal_images)

# Evaluate all three
whole_m = evaluate_new_embeddings(whole_emb, seg_y)
lung_m  = evaluate_new_embeddings(lung_emb, seg_y)
med_m   = evaluate_new_embeddings(med_emb, seg_y)

print(f'\n{"Method":20s} {"AUROC":>8s} {"TB Cov":>8s} {"Singleton":>10s}')
print('-' * 50)
for label, m in [('Whole image', whole_m), ('Lung only', lung_m), ('Mediastinal only', med_m)]:
    print(f'{label:20s} {m["auroc"]:8.4f} {m["tb_cov"]:8.4f} {m["singleton"]:10.4f}')

# Verify the results are actually different
assert whole_m != lung_m or whole_m != med_m, (
    'Whole-image and masked results are identical — masking had no effect. '
    'This should not happen with working lungmask segmentation.'
)

# Interpretation
delta_lung = lung_m['auroc'] - whole_m['auroc']
delta_med  = med_m['auroc'] - whole_m['auroc']
print(f'\nAUROC deltas vs whole-image:')
print(f'  Lung only:       {delta_lung:+.4f}')
print(f'  Mediastinal only: {delta_med:+.4f}')

if delta_lung >= 0.02:
    print('  -> Lung-only IMPROVES: whole-image may rely on non-lung shortcuts')
elif delta_lung <= -0.02:
    print('  -> Lung-only DEGRADES: contextual info carries TB signal')
else:
    print('  -> Minimal lung-only effect: model is not shortcut-dependent')

if med_m['auroc'] > 0.70:
    print(f'  -> Mediastinal AUROC {med_m["auroc"]:.3f}: non-lung features carry '
          f'substantial signal (potential shortcut concern)')
else:
    print(f'  -> Mediastinal AUROC {med_m["auroc"]:.3f}: non-lung features '
          f'have limited discriminative value')

In [ ]:
# ── Cell 9: Embedding cosine similarity analysis ─────────────────────
# How much do embeddings change when masking? High similarity → masking
# barely shifts the representation; low → model is sensitive to context.

from sklearn.metrics.pairwise import cosine_similarity

whole_norm = normalize(whole_emb, norm='l2')
lung_norm  = normalize(lung_emb, norm='l2')
med_norm   = normalize(med_emb, norm='l2')

# Per-image cosine similarity
cos_lung = np.array([cosine_similarity(whole_norm[i:i+1], lung_norm[i:i+1])[0, 0]
                      for i in range(n_seg)])
cos_med  = np.array([cosine_similarity(whole_norm[i:i+1], med_norm[i:i+1])[0, 0]
                      for i in range(n_seg)])

print(f'Cosine similarity (whole vs lung-only):')
print(f'  Mean: {cos_lung.mean():.4f}, Median: {np.median(cos_lung):.4f}, '
      f'Std: {cos_lung.std():.4f}')
print(f'Cosine similarity (whole vs mediastinal):')
print(f'  Mean: {cos_med.mean():.4f}, Median: {np.median(cos_med):.4f}, '
      f'Std: {cos_med.std():.4f}')

# Stratify by TB status
for label, mask in [('TB+', seg_y == 1), ('TB-', seg_y == 0)]:
    print(f'\n  {label}: cos(whole, lung) = {cos_lung[mask].mean():.4f}, '
          f'cos(whole, med) = {cos_med[mask].mean():.4f}')

In [ ]:
# ── Cell 10: Save results ────────────────────────────────────────────

seg_df = pd.DataFrame([
    {'method': 'whole_image',      'n': n_seg, 'seg_failures': seg_failures,
     'mean_lung_frac': round(np.nanmean(lung_fractions), 4),
     'cos_sim_vs_whole': 1.0, **whole_m},
    {'method': 'lung_only',        'n': n_seg, 'seg_failures': seg_failures,
     'mean_lung_frac': round(np.nanmean(lung_fractions), 4),
     'cos_sim_vs_whole': round(cos_lung.mean(), 4), **lung_m},
    {'method': 'mediastinal_only', 'n': n_seg, 'seg_failures': seg_failures,
     'mean_lung_frac': round(np.nanmean(lung_fractions), 4),
     'cos_sim_vs_whole': round(cos_med.mean(), 4), **med_m},
])

out_path = RESULTS_DIR / 'lung_segmentation.csv'
seg_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'  File size: {out_path.stat().st_size / 1e3:.1f} KB')
print()
print(seg_df.to_string(index=False))

In [ ]:
# ── Cell 11: Cleanup ─────────────────────────────────────────────────
del model, processor
del whole_emb, lung_emb, med_emb
del lung_only_images, mediastinal_images, seg_images, test_images
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Done. Copy lung_segmentation.csv to local results/tables/ directory.')